#### LangSmith 기본 예제


##### 1) 라이브러리 설치

In [ ]:
# uv add langsmith

##### 2) 환경 변수 설정 (Groq, LangSmith)

- `.env` 파일에 `GROQ_API_KEY`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` 등록
- `load_dotenv()`가 `.env` 값을 환경 변수로 설정하므로 LangSmith가 별도 코드 없이 읽어서 사용

In [1]:
import os
from dotenv import load_dotenv
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 필수 환경 변수 확인 (값은 출력하지 않고 설정 여부만 확인)
REQUIRED_ENV = ["GROQ_API_KEY", "LANGSMITH_API_KEY", "LANGSMITH_PROJECT"]
missing = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing:
    raise RuntimeError(f".env 파일에 다음 환경 변수가 없습니다: {', '.join(missing)}")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# LANGSMITH_TRACING 이 'true' 여야 트레이스가 LangSmith 로 전송됨
print("LangSmith 트레이싱:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

LangSmith 트레이싱: true
LangSmith 프로젝트: MyLangchainProject


##### LangSmith와 LangChain을 활용한 기본 로깅 예제

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langsmith import traceable

# LangSmith 설정(LANGSMITH_TRACING / API_KEY / PROJECT / ENDPOINT)은
# 위에서 load_dotenv() 로 환경 변수에 이미 로드되어 있으므로 다시 대입할 필요가 없음

# LLM 모델 설정 (Groq 사용: OpenAI 호환 API)
llm = ChatOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="openai/gpt-oss-120b",  
    temperature=0.7
)

# LangSmith로 실행 추적
@traceable(run_type="chain", name="Simple_Chain")
def ask_question(question: str):

    # 개별 메시지 템플릿 정의
    system_message = SystemMessagePromptTemplate.from_template(
        "당신은 유용한 AI 비서입니다."
    )
    user_message = HumanMessagePromptTemplate.from_template(
        "{question}"
    )

    chat_prompt = ChatPromptTemplate.from_messages([
        system_message,
        user_message,
    ])
    
    messages = chat_prompt.format_messages(question=question)
    response = llm.invoke(messages)

    return response.content

# 테스트 실행
question = "LangGraph와 LangChain의 차이점은 무엇인가요?"
answer = ask_question(question)

# 결과 출력
print("\n🔹 [AI 답변]:")
print(answer)



🔹 [AI 답변]:
## LangGraph와 LangChain의 핵심 차이점

| 구분 | **LangChain** | **LangGraph** |
|------|----------------|----------------|
| **주된 목적** | LLM‑기반 애플리케이션을 **체인(chain)**·**에이전트(agent)**·**메모리** 등으로 **구조화**하여 빠르게 프로토타입을 만들 수 있게 함 | **그래프 기반 워크플로우**를 통해 **상태(stateful) 흐름**·**조건·반복·분기** 등을 자연스럽게 모델링하고 실행 |
| **구조·패러다임** | *선형* 혹은 *단순 분기* 흐름을 `Chain`, `SequentialChain`, `RouterChain` 등으로 연결 | **노드(node) + 엣지(edge)** 로 이루어진 **유향 그래프**. 노드는 `Runnable`(LLM, Tool, Function 등)이고, 엣지는 데이터 흐름·조건을 정의 |
| **상태 관리** | 메모리(`ConversationBufferMemory`, `VectorStoreRetrieverMemory` 등)로 **대화 컨텍스트**를 유지. 상태는 체인 전체에 걸쳐 전역적이거나 체인 단위로 전달 | **노드별 상태**를 자동으로 추적. 그래프 실행 중 `State` 객체가 자동 업데이트돼, 이전 노드 결과를 다음 노드가 바로 참조 가능 |
| **조건·반복·루프** | `ConditionalRouter`, `LoopChain` 등 별도 구현 필요 → 복잡한 로직은 코드로 직접 작성 | 그래프 자체가 **조건 엣지**와 **루프 엣지**를 지원. `if/else`, `while`, `for` 같은 흐름을 선언형으로 정의 가능 |
| **디버깅·시각화** | 로그와 콜백을 통해 흐름을 추적하지만, 시각화 도구는 제한적 | `graph.draw()` 같은 메서드로 **실시간 DAG 시각화** 제공. 실행 트레이스와 상태 변화를 그래프 형태로 확인 가능 |
| **